# 05. Train Model Per Komoditas

Sesuai dengan review pipeline, notebook ini akan melatih model **LightGBM** secara individual untuk masing-masing dari 21 komoditas. Hyperparameter yang digunakan adalah hasil tuning terbaik yang disimpan pada `../models/per_komoditas_params.json`.

Model akan disimpan ke dalam `../api/models/per_komoditas/` agar bisa diakses langsung oleh API endpoint.

**Update**: Seleksi model otomatis menggunakan **RidgeCV** (alpha sweep: 0.1, 1, 10, 100, 1000) menggantikan Ridge alpha=10.0 fixed. Threshold seleksi diturunkan ke MAPE > 8% agar lebih banyak komoditas volatile masuk pertimbangan Ridge.

In [9]:
import os
import json
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

## (Opsional) Automatic Hyperparameter Tuning per Komoditas

Jalankan cell di bawah ini jika Anda ingin mencari hyperparameter terbaik secara otomatis menggunakan **RandomizedSearchCV** dan **TimeSeriesSplit** untuk mencegah overfitting, kemudian menyimpannya ke `../models/per_komoditas_params.json`.

In [10]:
# Set RUN_TUNING ke True jika ingin melatih/mencari ulang parameter terbaik
RUN_TUNING = False

if RUN_TUNING:
    from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
    print("Memulai proses hyperparameter tuning untuk 21 komoditas...")
    
    params_path = '../models/per_komoditas_params.json'
    try:
        with open(params_path, 'r') as f:
            tuned_params = json.load(f)
    except FileNotFoundError:
        tuned_params = {}

    # Definisikan ruang pencarian parameter yang cukup ketat/teregularisasi demi menghindari overfitting
    param_dist = {
        'max_depth': [3, 4, 5],
        'num_leaves': [7, 15, 20, 31],
        'learning_rate': [0.01, 0.03, 0.05, 0.1],
        'n_estimators': [50, 100, 150, 200],
        'min_child_samples': [10, 20, 30, 50],
        'reg_alpha': [0.0, 0.1, 0.5, 1.0],
        'reg_lambda': [0.5, 1.0, 2.0, 5.0],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0]
    }
    
    data_base_dir = '../data/processed/per_komoditas'
    if os.path.exists(data_base_dir):
        folders = [f for f in os.listdir(data_base_dir) if os.path.isdir(os.path.join(data_base_dir, f))]
        
        for folder in folders:
            komoditas_name = folder.replace('_', ' ').replace('-', '/')
            data_dir = f'{data_base_dir}/{folder}'
            
            try:
                X_train_k = pd.read_csv(f'{data_dir}/X_train.csv')
                y_train_k = pd.read_csv(f'{data_dir}/y_train.csv').squeeze()
            except FileNotFoundError:
                continue
                
            print(f"Tuning {komoditas_name}...")
            
            # Gunakan 3-fold TimeSeriesSplit karena data runtun waktu
            tscv = TimeSeriesSplit(n_splits=3)
            
            lgb_model = lgb.LGBMRegressor(random_state=42, verbose=-1)
            search = RandomizedSearchCV(
                estimator=lgb_model,
                param_distributions=param_dist,
                n_iter=15,
                cv=tscv,
                scoring='neg_mean_absolute_percentage_error',
                n_jobs=-1,
                random_state=42
            )
            
            search.fit(X_train_k, y_train_k)
            
            # Simpan hasil tuning terbaik
            best_p = search.best_params_
            tuned_params[komoditas_name] = best_p
            print(f"[TUNING OK] {komoditas_name} | Best CV MAPE: {-search.best_score_*100:.2f}%")
            
        # Tulis kembali hasil tuning ke file JSON
        os.makedirs(os.path.dirname(params_path), exist_ok=True)
        with open(params_path, 'w') as f:
            json.dump(tuned_params, f, indent=4)
        print(f"\nTuning selesai! Hyperparameter disimpan ke: {params_path}")
    else:
        print("[ERROR] Folder data/processed/per_komoditas tidak ditemukan.")
else:
    print("RUN_TUNING set ke False. Melewati proses automatic hyperparameter tuning.")

RUN_TUNING set ke False. Melewati proses automatic hyperparameter tuning.


## Load Hyperparameters

In [11]:
# Load best hyperparameters that were tuned previously
params_path = '../models/per_komoditas_params.json'
with open(params_path, 'r') as f:
    best_params = json.load(f)

print(f"Loaded hyperparameters for {len(best_params)} commodities from {params_path}.")

Loaded hyperparameters for 21 commodities from ../models/per_komoditas_params.json.


## Training Loop

Untuk setiap komoditas, notebook ini melatih tiga kandidat model:
1. **LightGBM** — dengan hyperparameter dari JSON + `random_state=42` konsisten
2. **RidgeCV** — alpha dipilih otomatis dari `[0.1, 1, 10, 100, 1000]` via 5-fold CV pada data train
3. **Naive (Lag-1)** — prediksi = harga bulan lalu (baseline)

Model terbaik dipilih berdasarkan MAPE test set. Seleksi Ridge/Naive hanya aktif jika `MAPE_LGBM > 8%`.

In [12]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import joblib

# ── Konstanta ──────────────────────────────────────────────────────────────────
FITUR = [
    'tahun', 'bulan', 'kuartal', 'wilayah_enc',
    'harga_lag1', 'harga_lag2', 'harga_lag3', 'harga_rolling3',
    'harga_lag12', 'sin_bulan', 'cos_bulan', 'is_lebaran_month',
    'tren_index', 'harga_rolling12'
]
TARGET = 'harga'

# Alpha candidates untuk RidgeCV — sweep lebar agar setiap komoditas punya alpha optimal
RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0, 1000.0]

# Threshold MAPE: jika LGBM di atas ini, pertimbangkan Ridge/Naive sebagai alternatif
# Diturunkan dari 10% ke 8% agar Cabai Merah Besar & Keriting (~22-24%) masuk seleksi
MAPE_THRESHOLD = 8.0

rekap_hasil = []

# Pastikan folder target ada
os.makedirs('../api/models/per_komoditas', exist_ok=True)


def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


for komoditas, params in best_params.items():
    folder_name = komoditas.replace(' ', '_').replace('/', '-')
    data_dir  = f'../data/processed/per_komoditas/{folder_name}'
    model_dir = f'../api/models/per_komoditas/{folder_name}'

    # ── Load dataset per komoditas ────────────────────────────────────────────
    try:
        X_train_k = pd.read_csv(f'{data_dir}/X_train.csv')
        X_test_k  = pd.read_csv(f'{data_dir}/X_test.csv')
        y_train_k = pd.read_csv(f'{data_dir}/y_train.csv').squeeze()
        y_test_k  = pd.read_csv(f'{data_dir}/y_test.csv').squeeze()
    except FileNotFoundError:
        print(f"[ERROR] Data untuk {komoditas} tidak ditemukan di {data_dir}. Skip.")
        continue

    # ── 1. LIGHTGBM ───────────────────────────────────────────────────────────
    lgb_params = {k: v for k, v in params.items() if k != 'model_type'}
    lgb_params['random_state'] = 42   # seed konsisten → hasil reproducible
    lgb_params['verbose'] = -1

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_k, y_train_k, test_size=0.15, shuffle=False
    )
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
    )
    pred_lgb = model_lgb.predict(X_test_k)
    mape_lgb = calculate_mape(y_test_k, pred_lgb)

    # ── 2. RIDGECV (alpha dipilih otomatis via CV pada train set) ─────────────
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_k)
    X_test_scaled  = scaler.transform(X_test_k)

    # RidgeCV pakai leave-one-out CV secara efisien jika cv=None
    # Kita pakai cv=5 agar lebih robust untuk time-series (data tidak di-shuffle)
    model_ridge = RidgeCV(alphas=RIDGE_ALPHAS, cv=5)
    model_ridge.fit(X_train_scaled, y_train_k)
    pred_ridge  = model_ridge.predict(X_test_scaled)
    mape_ridge  = calculate_mape(y_test_k, pred_ridge)

    # ── 3. NAIVE (Lag-1) ──────────────────────────────────────────────────────
    pred_naive = X_test_k['harga_lag1'].values
    mape_naive = calculate_mape(y_test_k, pred_naive)

    # ── Seleksi model terbaik ─────────────────────────────────────────────────
    best_model_name = 'lgbm'
    best_mape       = mape_lgb
    final_model     = model_lgb
    final_pred      = pred_lgb

    # Pertimbangkan Ridge & Naive jika LGBM melewati threshold
    if mape_lgb > MAPE_THRESHOLD:
        if mape_ridge < best_mape:
            best_model_name = 'ridge'
            best_mape       = mape_ridge
            final_model     = model_ridge
            final_pred      = pred_ridge

        if mape_naive < best_mape:
            best_model_name = 'naive'
            best_mape       = mape_naive
            final_model     = None
            final_pred      = pred_naive

    # Update params JSON dengan model_type yang dipakai
    best_params[komoditas]['model_type'] = best_model_name

    # ── Simpan model ke disk ──────────────────────────────────────────────────
    os.makedirs(model_dir, exist_ok=True)
    if best_model_name == 'lgbm':
        joblib.dump(final_model, f'{model_dir}/lgbm_model.joblib')
    elif best_model_name == 'ridge':
        joblib.dump(final_model, f'{model_dir}/ridge_model.joblib')
        joblib.dump(scaler,      f'{model_dir}/ridge_scaler.joblib')
    # naive: tidak ada model yang disimpan

    # ── Metrik final ──────────────────────────────────────────────────────────
    mae  = mean_absolute_error(y_test_k, final_pred)
    rmse = np.sqrt(mean_squared_error(y_test_k, final_pred))
    r2   = r2_score(y_test_k, final_pred)

    rekap_hasil.append({
        'Komoditas': komoditas,
        'Model':     best_model_name,
        'MAE':       mae,
        'RMSE':      rmse,
        'R2':        r2,
        'MAPE(%)':   best_mape,
        # Simpan juga perbandingan ketiga model untuk analisis
        'MAPE_LGBM(%)':  round(mape_lgb, 4),
        'MAPE_Ridge(%)': round(mape_ridge, 4),
        'MAPE_Naive(%)': round(mape_naive, 4),
        'Ridge_alpha':   round(model_ridge.alpha_, 4),
    })

    # Info alpha Ridge untuk debug
    ridge_info = f"(α={model_ridge.alpha_:.1f})"
    print(
        f"[{best_model_name.upper():5s}] {komoditas:40s} "
        f"| LGBM:{mape_lgb:6.2f}% Ridge:{mape_ridge:6.2f}% {ridge_info} Naive:{mape_naive:6.2f}% "
        f"→ BEST: {best_mape:.2f}%"
    )

# ── Simpan params JSON yang sudah diupdate model_type ─────────────────────────
# Simpan params ke models/ (sumber notebook)
with open('../models/per_komoditas_params.json', 'w') as f:
    json.dump(best_params, f, indent=4)

# PENTING: Copy juga ke api/models/ agar FastAPI dapat membacanya
os.makedirs('../api/models', exist_ok=True)
import shutil as _shutil
_shutil.copy('../models/per_komoditas_params.json', '../api/models/per_komoditas_params.json')
print('per_komoditas_params.json berhasil disalin ke api/models/')

# ── Simpan rekap hasil training ke CSV ────────────────────────────────────────
df_rekap = pd.DataFrame(rekap_hasil)
# Kolom utama dulu, kolom perbandingan di belakang
cols_utama = ['Komoditas', 'Model', 'MAE', 'RMSE', 'R2', 'MAPE(%)']
cols_extra = ['MAPE_LGBM(%)', 'MAPE_Ridge(%)', 'MAPE_Naive(%)', 'Ridge_alpha']
df_rekap[cols_utama].to_csv('../data/processed/rekap_hasil_per_komoditas.csv', index=False)
df_rekap[cols_utama + cols_extra].to_csv('../data/processed/rekap_perbandingan_model.csv', index=False)

print("\nTraining selesai!")
print("  Rekap utama   : ../data/processed/rekap_hasil_per_komoditas.csv")
print("  Perbandingan  : ../data/processed/rekap_perbandingan_model.csv")
print()
print(df_rekap[['Komoditas','Model','MAPE(%)','MAPE_LGBM(%)','MAPE_Ridge(%)','Ridge_alpha']].to_string(index=False))

[RIDGE] Bawang Merah Ukuran Sedang               | LGBM: 14.16% Ridge: 12.17% (α=1.0) Naive:100.00% → BEST: 12.17%
[LGBM ] Bawang Putih Ukuran Sedang               | LGBM:  5.78% Ridge:  5.71% (α=0.1) Naive:100.00% → BEST: 5.78%
[LGBM ] Beras Kualitas Bawah I                   | LGBM:  2.11% Ridge:  1.75% (α=1.0) Naive: 99.99% → BEST: 2.11%
[LGBM ] Beras Kualitas Bawah II                  | LGBM:  3.24% Ridge:  1.57% (α=0.1) Naive: 99.99% → BEST: 3.24%
[LGBM ] Beras Kualitas Medium I                  | LGBM:  1.93% Ridge:  1.72% (α=1.0) Naive: 99.99% → BEST: 1.93%
[LGBM ] Beras Kualitas Medium II                 | LGBM:  2.03% Ridge:  1.73% (α=0.1) Naive: 99.99% → BEST: 2.03%
[LGBM ] Beras Kualitas Super I                   | LGBM:  3.06% Ridge:  2.18% (α=1.0) Naive: 99.99% → BEST: 3.06%
[LGBM ] Beras Kualitas Super II                  | LGBM:  2.85% Ridge:  2.19% (α=1.0) Naive: 99.99% → BEST: 2.85%
[LGBM ] Cabai Merah Besar                        | LGBM: 22.23% Ridge: 23.62% (α=1.0) N